# ⚡ LNG Deal Financial Feasibility Analyser

**End-to-End Deal Evaluation Tool** — From origination through settlement

---
### How to Use
1. **Run all cells** (`Kernel → Restart & Run All`)
2. **Adjust sliders and inputs** in each section
3. The model **recalculates automatically**
4. Review the **Go / No-Go verdict** at the bottom

> 💡 All inputs have realistic defaults based on current LNG market conditions.
---

In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 0 — ENVIRONMENT SETUP (Colab + Local Jupyter)
# ─────────────────────────────────────────────────────────────────────────────

import subprocess, sys, shutil

# ── Detect environment ────────────────────────────────────────────────────────
try:
    from google.colab import files as colab_files, output as colab_output
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ── Install missing packages ──────────────────────────────────────────────────
for pkg in ['ipywidgets', 'scipy']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

# ── Enable widgets for local Jupyter ─────────────────────────────────────────
if not IN_COLAB:
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                               'widgetsnbextension', 'jupyterlab-widgets'])
        # Enable for classic notebook (harmless if already enabled)
        if shutil.which('jupyter'):
            subprocess.run(['jupyter', 'nbextension', 'enable', '--py',
                            'widgetsnbextension', '--sys-prefix'],
                           capture_output=True)
    except Exception:
        pass  # Non-critical — widgets may already work

print('✅  Environment ready.', '(Colab)' if IN_COLAB else '(Local Jupyter)')
if not IN_COLAB:
    print('   ℹ️  If sliders don\'t appear, restart the notebook server after this first run.')

✅  Environment ready. (Local Jupyter)
   ℹ️  If sliders don't appear, restart the notebook server after this first run.


In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 — IMPORTS & CONFIGURATION
# Run this cell first to load all required libraries
# ─────────────────────────────────────────────────────────────────────────────

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mtick
from matplotlib.gridspec import GridSpec
from matplotlib.patches import FancyBboxPatch
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from scipy import stats
from datetime import date, timedelta
import json

%matplotlib inline
plt.rcParams.update({
    'figure.facecolor':  '#0d1117',
    'axes.facecolor':    '#161b22',
    'axes.edgecolor':    '#30363d',
    'axes.labelcolor':   '#e6edf3',
    'xtick.color':       '#8b949e',
    'ytick.color':       '#8b949e',
    'text.color':        '#e6edf3',
    'grid.color':        '#21262d',
    'grid.linestyle':    '--',
    'grid.alpha':        0.6,
    'font.family':       'DejaVu Sans',
    'font.size':         10,
    'axes.titlesize':    12,
    'axes.titleweight':  'bold',
    'figure.dpi':        110,
})

# ── Colour palette ────────────────────────────────────────────────────────────
C = {
    'gain':    '#3fb950',   # green
    'loss':    '#f85149',   # red
    'warn':    '#d29922',   # amber
    'blue':    '#58a6ff',   # accent blue
    'purple':  '#bc8cff',   # purple
    'teal':    '#39d353',   # teal
    'bg':      '#0d1117',   # page bg
    'card':    '#161b22',   # card bg
    'border':  '#30363d',   # border
    'text':    '#e6edf3',   # primary text
    'muted':   '#8b949e',   # secondary text
}

# ── Shared output widget ──────────────────────────────────────────────────────
main_out = widgets.Output()

print('✅  Libraries loaded. Proceed to Section 1 below.')

✅  Libraries loaded. Proceed to Section 1 below.


---
## 📦 Section 1 — Cargo & Volume Parameters
*Define the physical LNG cargo — vessel size, density, energy content, and boil-off gas.*

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 — CARGO & VOLUME INPUTS
# ─────────────────────────────────────────────────────────────────────────────

style  = {'description_width': '200px'}
layout = widgets.Layout(width='480px')

# ── Cargo inputs ──────────────────────────────────────────────────────────────
w_volume = widgets.FloatSlider(
    value=138000, min=50000, max=270000, step=1000,
    description='Vessel Capacity (m³):',
    style=style, layout=layout,
    readout_format='.0f'
)
w_density = widgets.FloatSlider(
    value=0.46, min=0.42, max=0.50, step=0.001,
    description='LNG Density (t/m³):',
    style=style, layout=layout,
    readout_format='.3f'
)
w_cv = widgets.FloatSlider(
    value=21.50, min=19.0, max=23.5, step=0.05,
    description='Calorific Value (MMBtu/t):',
    style=style, layout=layout,
    readout_format='.2f'
)
w_bog_rate = widgets.FloatSlider(
    value=0.10, min=0.05, max=0.20, step=0.005,
    description='BOG Rate (%/day):',
    style=style, layout=layout,
    readout_format='.3f'
)
w_voyage_days = widgets.IntSlider(
    value=20, min=5, max=35, step=1,
    description='Voyage Days (load→disch):',
    style=style, layout=layout
)
w_heel = widgets.FloatSlider(
    value=1000, min=500, max=3000, step=100,
    description='Heel Retained (m³):',
    style=style, layout=layout,
    readout_format='.0f'
)

cargo_box = widgets.VBox([
    widgets.HTML('<b style="color:#58a6ff;font-size:13px">▸ Physical Cargo Parameters</b>'),
    w_volume, w_density, w_cv,
    widgets.HTML('<b style="color:#58a6ff;font-size:13px">▸ Voyage & BOG</b>'),
    w_bog_rate, w_voyage_days, w_heel
], layout=widgets.Layout(padding='12px', border='1px solid #30363d', border_radius='8px'))

# ── Live volume summary ───────────────────────────────────────────────────────
vol_out = widgets.Output()

def calc_volumes(vol, dens, cv, bog, days, heel):
    weight        = vol * dens
    gross_mmbtu   = weight * cv
    bog_lost      = gross_mmbtu * (bog/100) * days
    heel_mmbtu    = heel * dens * cv
    net_delivered = gross_mmbtu - bog_lost - heel_mmbtu
    return dict(
        weight=weight, gross=gross_mmbtu,
        bog=bog_lost, heel=heel_mmbtu,
        net=net_delivered,
        bog_pct=bog_lost/gross_mmbtu*100
    )

def refresh_volumes(*_):
    v = calc_volumes(
        w_volume.value, w_density.value, w_cv.value,
        w_bog_rate.value, w_voyage_days.value, w_heel.value
    )
    with vol_out:
        clear_output(wait=True)
        print(f"""
╔══════════════════════════════════════╗
║   CARGO VOLUME SUMMARY               ║
╠══════════════════════════════════════╣
║  Weight:           {v['weight']:>10,.0f}  t  ║
║  Gross Energy:     {v['gross']:>10,.0f} MMBtu║
║  BOG Loss:         {v['bog']:>10,.0f} MMBtu║
║  BOG Loss (%):     {v['bog_pct']:>9.2f}%      ║
║  Heel Retained:    {v['heel']:>10,.0f} MMBtu║
║  ─────────────────────────────────  ║
║  NET DELIVERED:    {v['net']:>10,.0f} MMBtu║
╚══════════════════════════════════════╝""")

for w in [w_volume, w_density, w_cv, w_bog_rate,
          w_voyage_days, w_heel]:
    w.observe(refresh_volumes, names='value')

display(widgets.HBox([cargo_box, vol_out]))
refresh_volumes()

---
## 💰 Section 2 — Pricing Formula
*Configure the buy and sell pricing — JCC slope, JKM spot, TTF, or fixed price.*

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 — PRICING INPUTS
# ─────────────────────────────────────────────────────────────────────────────

# ── BUY SIDE ─────────────────────────────────────────────────────────────────
w_buy_type = widgets.Dropdown(
    options=['JCC Slope', 'JKM Spot', 'TTF Linked', 'Henry Hub', 'Fixed Price'],
    value='JCC Slope',
    description='Buy Price Type:', style=style, layout=layout
)
w_jcc = widgets.FloatSlider(
    value=86.40, min=30.0, max=130.0, step=0.50,
    description='JCC Price ($/bbl):', style=style, layout=layout,
    readout_format='.2f'
)
w_slope = widgets.FloatSlider(
    value=0.1485, min=0.10, max=0.18, step=0.001,
    description='JCC Slope:', style=style, layout=layout,
    readout_format='.4f'
)
w_constant = widgets.FloatText(
    value=0.50, description='Formula Constant ($/MMBtu):',
    style=style, layout=layout
)
w_scurve_cap = widgets.FloatSlider(
    value=100.0, min=70.0, max=150.0, step=1.0,
    description='S-Curve Cap ($/bbl):', style=style, layout=layout,
    readout_format='.1f'
)
w_scurve_floor = widgets.FloatSlider(
    value=40.0, min=20.0, max=70.0, step=1.0,
    description='S-Curve Floor ($/bbl):', style=style, layout=layout,
    readout_format='.1f'
)
w_jkm_buy = widgets.FloatSlider(
    value=13.80, min=4.0, max=50.0, step=0.05,
    description='JKM Spot ($/MMBtu):', style=style, layout=layout,
    readout_format='.2f'
)
w_buy_discount = widgets.FloatText(
    value=-0.52, description='Buy Discount ($/MMBtu):',
    style=style, layout=layout
)
w_hh = widgets.FloatSlider(
    value=3.50, min=1.5, max=12.0, step=0.05,
    description='Henry Hub ($/MMBtu):', style=style, layout=layout,
    readout_format='.2f'
)
w_hh_mult = widgets.FloatSlider(
    value=1.15, min=1.00, max=1.30, step=0.01,
    description='HH Multiplier:', style=style, layout=layout,
    readout_format='.2f'
)
w_liq_fee = widgets.FloatText(
    value=3.00, description='Liquefaction Fee ($/MMBtu):',
    style=style, layout=layout
)
w_fixed_buy = widgets.FloatText(
    value=13.00, description='Fixed Buy Price ($/MMBtu):',
    style=style, layout=layout
)

# ── SELL SIDE ─────────────────────────────────────────────────────────────────
w_sell_type = widgets.Dropdown(
    options=['JKM Spot', 'TTF Linked', 'JCC Slope', 'Fixed Price'],
    value='JKM Spot',
    description='Sell Price Type:', style=style, layout=layout
)
w_jkm_sell = widgets.FloatSlider(
    value=14.20, min=4.0, max=50.0, step=0.05,
    description='JKM Sell ($/MMBtu):', style=style, layout=layout,
    readout_format='.2f'
)
w_sell_premium = widgets.FloatText(
    value=-0.30, description='Sell Premium/Discount:',
    style=style, layout=layout
)
w_ttf_sell = widgets.FloatSlider(
    value=16.50, min=4.0, max=60.0, step=0.05,
    description='TTF Sell ($/MMBtu):', style=style, layout=layout,
    readout_format='.2f'
)
w_ttf_premium = widgets.FloatText(
    value=-0.30, description='TTF Premium/Discount:',
    style=style, layout=layout
)
w_fixed_sell = widgets.FloatText(
    value=14.00, description='Fixed Sell Price ($/MMBtu):',
    style=style, layout=layout
)

# ── Dynamic panel switching ────────────────────────────────────────────────────
buy_panel  = widgets.VBox([])
sell_panel = widgets.VBox([])
price_out  = widgets.Output()

def get_buy_price():
    bt = w_buy_type.value
    if bt == 'JCC Slope':
        jcc_adj = np.clip(w_jcc.value, w_scurve_floor.value, w_scurve_cap.value)
        return w_slope.value * jcc_adj + w_constant.value
    elif bt == 'JKM Spot':
        return w_jkm_buy.value + w_buy_discount.value
    elif bt == 'TTF Linked':
        return w_ttf_sell.value + w_buy_discount.value
    elif bt == 'Henry Hub':
        return w_hh_mult.value * w_hh.value + w_liq_fee.value
    else:
        return w_fixed_buy.value

def get_sell_price():
    st = w_sell_type.value
    if st == 'JKM Spot':
        return w_jkm_sell.value + w_sell_premium.value
    elif st == 'TTF Linked':
        return w_ttf_sell.value + w_ttf_premium.value
    elif st == 'JCC Slope':
        jcc_adj = np.clip(w_jcc.value, w_scurve_floor.value, w_scurve_cap.value)
        return w_slope.value * jcc_adj + w_constant.value + w_sell_premium.value
    else:
        return w_fixed_sell.value

def update_buy_panel(*_):
    bt = w_buy_type.value
    if bt == 'JCC Slope':
        buy_panel.children = [w_jcc, w_slope, w_constant, w_scurve_cap, w_scurve_floor]
    elif bt == 'JKM Spot':
        buy_panel.children = [w_jkm_buy, w_buy_discount]
    elif bt == 'TTF Linked':
        buy_panel.children = [w_ttf_sell, w_buy_discount]
    elif bt == 'Henry Hub':
        buy_panel.children = [w_hh, w_hh_mult, w_liq_fee]
    else:
        buy_panel.children = [w_fixed_buy]
    refresh_price()

def update_sell_panel(*_):
    st = w_sell_type.value
    if st == 'JKM Spot':
        sell_panel.children = [w_jkm_sell, w_sell_premium]
    elif st == 'TTF Linked':
        sell_panel.children = [w_ttf_sell, w_ttf_premium]
    elif st == 'JCC Slope':
        sell_panel.children = [w_jcc, w_slope, w_constant, w_sell_premium]
    else:
        sell_panel.children = [w_fixed_sell]
    refresh_price()

def refresh_price(*_):
    bp = get_buy_price()
    sp = get_sell_price()
    margin = sp - bp
    with price_out:
        clear_output(wait=True)
        arrow = '▲' if margin >= 0 else '▼'
        color = '\033[92m' if margin >= 0 else '\033[91m'
        reset = '\033[0m'
        print(f"""
╔══════════════════════════════╗
║   INDICATIVE PRICES          ║
╠══════════════════════════════╣
║  Buy Price:   ${bp:>8.4f}/MMBtu ║
║  Sell Price:  ${sp:>8.4f}/MMBtu ║
║  ─────────────────────────  ║
║  Raw Margin:  ${margin:>+8.4f}/MMBtu ║
╚══════════════════════════════╝""")

for w in [w_buy_type, w_sell_type, w_jcc, w_slope, w_constant,
          w_scurve_cap, w_scurve_floor, w_jkm_buy, w_buy_discount,
          w_hh, w_hh_mult, w_liq_fee, w_fixed_buy,
          w_jkm_sell, w_sell_premium, w_ttf_sell,
          w_ttf_premium, w_fixed_sell]:
    w.observe(refresh_price, names='value')

w_buy_type.observe(update_buy_panel, names='value')
w_sell_type.observe(update_sell_panel, names='value')

buy_box = widgets.VBox([
    widgets.HTML('<b style="color:#3fb950;font-size:13px">▸ BUY SIDE (Purchase)</b>'),
    w_buy_type, buy_panel
], layout=widgets.Layout(padding='12px', border='1px solid #30363d',
                          border_radius='8px', width='500px'))

sell_box = widgets.VBox([
    widgets.HTML('<b style="color:#f85149;font-size:13px">▸ SELL SIDE (Sale)</b>'),
    w_sell_type, sell_panel
], layout=widgets.Layout(padding='12px', border='1px solid #30363d',
                          border_radius='8px', width='500px'))

update_buy_panel()
update_sell_panel()

display(widgets.HBox([buy_box, widgets.VBox([price_out]), sell_box],
                     layout=widgets.Layout(gap='16px')))

---
## 🚢 Section 3 — Shipping, Terminal & Cost Structure
*All voyage, port, terminal and other costs associated with the cargo.*

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 — COST INPUTS
# ─────────────────────────────────────────────────────────────────────────────

w_freight = widgets.FloatSlider(
    value=2.45, min=0.5, max=6.0, step=0.05,
    description='Voyage Charter ($/MMBtu):', style=style, layout=layout,
    readout_format='.2f'
)
w_load_port = widgets.FloatText(
    value=120000, description='Load Port Dues ($):',
    style=style, layout=layout
)
w_disch_port = widgets.FloatText(
    value=150000, description='Discharge Port Dues ($):',
    style=style, layout=layout
)
w_canal = widgets.FloatText(
    value=0, description='Canal Dues ($, 0 if none):',
    style=style, layout=layout
)
w_insurance = widgets.FloatText(
    value=350000, description='Marine Insurance ($):',
    style=style, layout=layout
)
w_ballast = widgets.FloatText(
    value=180000, description='Ballast Bonus ($):',
    style=style, layout=layout
)
w_regas = widgets.FloatSlider(
    value=0.40, min=0.0, max=1.50, step=0.01,
    description='Regasification ($/MMBtu):', style=style, layout=layout,
    readout_format='.2f'
)
w_storage = widgets.FloatSlider(
    value=0.05, min=0.0, max=0.50, step=0.01,
    description='Storage Fee ($/MMBtu):', style=style, layout=layout,
    readout_format='.2f'
)
w_brokerage = widgets.FloatSlider(
    value=0.03, min=0.0, max=0.20, step=0.005,
    description='Brokerage ($/MMBtu):', style=style, layout=layout,
    readout_format='.3f'
)
w_other = widgets.FloatText(
    value=50000, description='Other Costs ($):',
    style=style, layout=layout
)

# ── Finance inputs ────────────────────────────────────────────────────────────
w_discount_rate = widgets.FloatSlider(
    value=5.0, min=1.0, max=15.0, step=0.25,
    description='Discount Rate (% p.a.):', style=style, layout=layout,
    readout_format='.2f'
)
w_payment_days = widgets.IntSlider(
    value=5, min=0, max=30, step=1,
    description='Payment Terms (days):', style=style, layout=layout
)
w_fx = widgets.FloatText(
    value=1.0, description='FX Rate (to USD):',
    style=style, layout=layout
)
w_tax_rate = widgets.FloatSlider(
    value=17.0, min=0.0, max=35.0, step=0.5,
    description='Tax Rate (%):', style=style, layout=layout,
    readout_format='.1f'
)

cost_box = widgets.VBox([
    widgets.HTML('<b style="color:#d29922;font-size:13px">▸ Shipping & Logistics</b>'),
    w_freight, w_load_port, w_disch_port, w_canal, w_insurance, w_ballast,
    widgets.HTML('<b style="color:#d29922;font-size:13px">▸ Terminal Costs</b>'),
    w_regas, w_storage,
    widgets.HTML('<b style="color:#d29922;font-size:13px">▸ Other Costs</b>'),
    w_brokerage, w_other,
], layout=widgets.Layout(padding='12px', border='1px solid #30363d',
                          border_radius='8px', width='520px'))

fin_box = widgets.VBox([
    widgets.HTML('<b style="color:#bc8cff;font-size:13px">▸ Finance & Tax</b>'),
    w_discount_rate, w_payment_days, w_fx, w_tax_rate,
], layout=widgets.Layout(padding='12px', border='1px solid #30363d',
                          border_radius='8px', width='520px'))

display(widgets.HBox([cost_box, fin_box],
                     layout=widgets.Layout(gap='16px')))

---
## 📊 Section 4 — Risk Parameters
*Hedge ratio, VaR confidence, stress test shocks and credit parameters.*

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5 — RISK INPUTS
# ─────────────────────────────────────────────────────────────────────────────

w_hedge_ratio = widgets.FloatSlider(
    value=80.0, min=0.0, max=100.0, step=5.0,
    description='Hedge Ratio (%):', style=style, layout=layout,
    readout_format='.0f'
)
w_hedge_price = widgets.FloatText(
    value=0.0,
    description='Hedge Fixed Price ($/MMBtu, 0=at market):',
    style=style, layout=layout
)
w_price_vol = widgets.FloatSlider(
    value=2.5, min=0.5, max=8.0, step=0.1,
    description='Daily Price Vol (%):', style=style, layout=layout,
    readout_format='.1f'
)
w_var_conf = widgets.SelectionSlider(
    options=[90, 95, 99], value=95,
    description='VaR Confidence (%):', style=style, layout=layout
)
w_var_horizon = widgets.IntSlider(
    value=1, min=1, max=10, step=1,
    description='VaR Horizon (days):', style=style, layout=layout
)
w_credit_limit = widgets.FloatText(
    value=50000000, description='Counterparty Credit Limit ($):',
    style=style, layout=layout
)
w_stress_jkm = widgets.FloatSlider(
    value=-3.0, min=-20.0, max=20.0, step=0.5,
    description='Stress: JKM Shock ($/MMBtu):', style=style, layout=layout,
    readout_format='.1f'
)
w_stress_freight = widgets.FloatSlider(
    value=0.5, min=-2.0, max=3.0, step=0.1,
    description='Stress: Freight Shock ($/MMBtu):', style=style, layout=layout,
    readout_format='.1f'
)
w_stress_fx = widgets.FloatSlider(
    value=0.0, min=-15.0, max=15.0, step=0.5,
    description='Stress: FX Move (%):', style=style, layout=layout,
    readout_format='.1f'
)

risk_box = widgets.VBox([
    widgets.HTML('<b style="color:#f85149;font-size:13px">▸ Hedge Configuration</b>'),
    w_hedge_ratio, w_hedge_price,
    widgets.HTML('<b style="color:#f85149;font-size:13px">▸ Market Risk (VaR)</b>'),
    w_price_vol, w_var_conf, w_var_horizon,
    widgets.HTML('<b style="color:#f85149;font-size:13px">▸ Credit & Stress</b>'),
    w_credit_limit, w_stress_jkm, w_stress_freight, w_stress_fx,
], layout=widgets.Layout(padding='12px', border='1px solid #30363d',
                          border_radius='8px'))

display(risk_box)

---
## 🧮 Section 5 — Full Analysis Engine
*Run this cell to compute all financial metrics and display the complete feasibility report.*

In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 6 — ANALYSIS ENGINE
# ─────────────────────────────────────────────────────────────────────────────

def compute_all():
    """Master calculation function — returns full results dict."""

    # ── 1. Volumes ────────────────────────────────────────────────────────────
    v = calc_volumes(
        w_volume.value, w_density.value, w_cv.value,
        w_bog_rate.value, w_voyage_days.value, w_heel.value
    )
    net = v['net']

    # ── 2. Prices ─────────────────────────────────────────────────────────────
    buy_px   = get_buy_price()
    sell_px  = get_sell_price()
    fx       = w_fx.value if w_fx.value > 0 else 1.0
    sell_usd = sell_px * fx

    # ── 3. Revenue & COGS ─────────────────────────────────────────────────────
    revenue  = net * sell_usd
    cogs     = v['gross'] * buy_px          # pay for gross loaded
    phys_margin = revenue - cogs

    # ── 4. Costs ──────────────────────────────────────────────────────────────
    freight_var  = w_freight.value * net
    freight_fix  = (w_load_port.value + w_disch_port.value
                    + w_canal.value + w_insurance.value
                    + w_ballast.value)
    total_freight = freight_var + freight_fix

    regas_cost   = w_regas.value * net
    storage_cost = w_storage.value * net
    broker_cost  = w_brokerage.value * net
    other_cost   = w_other.value
    bog_cost     = v['bog'] * buy_px

    total_costs  = (total_freight + regas_cost + storage_cost
                    + broker_cost + other_cost + bog_cost)

    # ── 5. Hedge P&L ──────────────────────────────────────────────────────────
    hedge_vol   = net * (w_hedge_ratio.value / 100)
    hedge_px    = w_hedge_price.value if w_hedge_price.value > 0 \
                  else sell_px  # at market
    # Simplified: if sell drops vs hedge → gain; if rises → loss
    # For feasibility we show hedge P&L as incremental vs unhedged
    hedge_gain  = (hedge_px - sell_px) * hedge_vol

    # ── 6. Operating profit ───────────────────────────────────────────────────
    ebitda       = phys_margin - total_costs
    tax_amount   = max(0, ebitda * w_tax_rate.value / 100)
    net_profit   = ebitda - tax_amount + hedge_gain

    # ── 7. Per MMBtu metrics ──────────────────────────────────────────────────
    margin_mmbtu = net_profit / net if net > 0 else 0
    breakeven    = (cogs + total_costs) / net if net > 0 else 0

    # ── 8. VaR ────────────────────────────────────────────────────────────────
    daily_vol_pct = w_price_vol.value / 100
    z = {'90': 1.282, '95': 1.645, '99': 2.326}
    z_val = z[str(w_var_conf.value)]
    unhedged_vol  = net * (1 - w_hedge_ratio.value / 100)
    var_1d = z_val * daily_vol_pct * sell_px * unhedged_vol
    var_nd = var_1d * np.sqrt(w_var_horizon.value)

    # ── 9. Sensitivity analysis ───────────────────────────────────────────────
    price_range = np.linspace(sell_px * 0.5, sell_px * 1.6, 60)
    def pnl_at_price(p):
        rev = net * p * fx
        h_g = (hedge_px - p) * hedge_vol
        op  = rev - cogs - total_costs + h_g
        return op - max(0, op * w_tax_rate.value / 100)
    pnl_curve = [pnl_at_price(p) for p in price_range]

    # ── 10. Break-even analysis ───────────────────────────────────────────────
    breakeven_sell = (cogs + total_costs) / net if net > 0 else 0
    be_margin = sell_usd - breakeven_sell  # $/MMBtu headroom

    # ── 11. Stress test ───────────────────────────────────────────────────────
    s_sell  = sell_px + w_stress_jkm.value
    s_frt   = w_freight.value + w_stress_freight.value
    s_fx    = fx * (1 + w_stress_fx.value / 100)
    s_rev   = net * s_sell * s_fx
    s_frt_t = s_frt * net + freight_fix
    s_costs = (s_frt_t + regas_cost + storage_cost
               + broker_cost + other_cost + bog_cost)
    s_op    = s_rev - cogs - s_costs
    s_tax   = max(0, s_op * w_tax_rate.value / 100)
    s_net   = s_op - s_tax
    stress_delta = s_net - net_profit

    # ── 12. Credit utilization ────────────────────────────────────────────────
    credit_util = (revenue / w_credit_limit.value * 100
                   if w_credit_limit.value > 0 else 0)

    # ── 13. NPV / payback ─────────────────────────────────────────────────────
    r = w_discount_rate.value / 100
    t_years = (w_voyage_days.value + w_payment_days.value) / 365
    npv = net_profit / ((1 + r) ** t_years)

    # ── 14. Working capital (peak cash out) ───────────────────────────────────
    peak_cash = cogs + total_freight + regas_cost

    # ── 15. Return on capital ─────────────────────────────────────────────────
    roc = (net_profit / peak_cash * 100) if peak_cash > 0 else 0

    # ── 16. S-Curve visualization data ───────────────────────────────────────
    jcc_range = np.linspace(20, 130, 200)
    def scurve_price(jcc):
        jcc_adj = np.clip(jcc, w_scurve_floor.value, w_scurve_cap.value)
        return w_slope.value * jcc_adj + w_constant.value
    formula_price = [w_slope.value * j + w_constant.value for j in jcc_range]
    scurve_p      = [scurve_price(j) for j in jcc_range]

    return dict(
        # Volumes
        v=v, net=net,
        # Prices
        buy_px=buy_px, sell_px=sell_px, sell_usd=sell_usd,
        breakeven_sell=breakeven_sell, be_margin=be_margin,
        # P&L components
        revenue=revenue, cogs=cogs,
        phys_margin=phys_margin,
        freight_var=freight_var, freight_fix=freight_fix,
        total_freight=total_freight,
        regas_cost=regas_cost, storage_cost=storage_cost,
        broker_cost=broker_cost, other_cost=other_cost,
        bog_cost=bog_cost, total_costs=total_costs,
        hedge_gain=hedge_gain, hedge_px=hedge_px,
        ebitda=ebitda, tax_amount=tax_amount,
        net_profit=net_profit, margin_mmbtu=margin_mmbtu,
        npv=npv, roc=roc, peak_cash=peak_cash,
        # Risk
        var_1d=var_1d, var_nd=var_nd, unhedged_vol=unhedged_vol,
        # Sensitivity
        price_range=price_range, pnl_curve=pnl_curve,
        # Stress
        s_net=s_net, stress_delta=stress_delta,
        # Credit
        credit_util=credit_util,
        # S-Curve
        jcc_range=jcc_range,
        formula_price=formula_price,
        scurve_p=scurve_p,
    )

print('✅  Analysis engine ready. Proceed to Section 6 to generate the report.')

✅  Analysis engine ready. Proceed to Section 6 to generate the report.


---
## 📈 Section 6 — Full Feasibility Report
*Click **Run Full Analysis** to generate the complete financial report and charts.*

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 7 — REPORT GENERATION
# ─────────────────────────────────────────────────────────────────────────────

report_out = widgets.Output()

def generate_report(_=None):
    r = compute_all()

    # ── Verdict ───────────────────────────────────────────────────────────────
    if r['net_profit'] > 0:
        verdict, vcolor = '✅  FEASIBLE — GO', C['gain']
    elif r['net_profit'] > -r['peak_cash'] * 0.02:
        verdict, vcolor = '⚠️  MARGINAL — REVIEW', C['warn']
    else:
        verdict, vcolor = '❌  NOT FEASIBLE — NO GO', C['loss']

    with report_out:
        clear_output(wait=True)

        # ══════════════════════════════════════════════════════════════════════
        # FIG 1 — KPI DASHBOARD (top banner)
        # ══════════════════════════════════════════════════════════════════════
        fig1, axes = plt.subplots(1, 6, figsize=(18, 2.2))
        fig1.patch.set_facecolor(C['bg'])

        kpis = [
            ('NET P&L',       r['net_profit'],   '${:,.0f}',  r['net_profit'] >= 0),
            ('P&L / MMBtu',   r['margin_mmbtu'], '${:+.3f}',  r['margin_mmbtu'] >= 0),
            ('B/E Sell Px',   r['breakeven_sell'],'${:.3f}',  True),
            ('VaR (1d)',      -r['var_1d'],      '${:,.0f}',  False),
            ('ROC',           r['roc'],          '{:+.1f}%',  r['roc'] >= 0),
            ('Credit Util',   r['credit_util'],  '{:.1f}%',
             r['credit_util'] < 80),
        ]

        for ax, (lbl, val, fmt, is_good) in zip(axes, kpis):
            ax.set_facecolor('#161b22')
            for spine in ax.spines.values():
                spine.set_edgecolor(C['border'])
                spine.set_linewidth(0.8)
            ax.set_xticks([]); ax.set_yticks([])

            color = C['gain'] if is_good else C['loss']
            if lbl in ('B/E Sell Px', 'VaR (1d)'):
                color = C['blue']
            if lbl == 'Credit Util':
                color = C['gain'] if val < 80 else (
                    C['warn'] if val < 100 else C['loss']
                )

            ax.text(0.5, 0.72, fmt.format(val),
                    ha='center', va='center',
                    fontsize=15, fontweight='bold',
                    color=color,
                    transform=ax.transAxes)
            ax.text(0.5, 0.22, lbl,
                    ha='center', va='center',
                    fontsize=8, color=C['muted'],
                    fontweight='bold',
                    transform=ax.transAxes)

        fig1.suptitle(f'LNG DEAL FEASIBILITY — {verdict}',
                      fontsize=14, fontweight='bold',
                      color=vcolor, y=1.06)
        plt.tight_layout()
        plt.show()

        # ══════════════════════════════════════════════════════════════════════
        # FIG 2 — WATERFALL + SENSITIVITY (main analysis)
        # ══════════════════════════════════════════════════════════════════════
        fig2 = plt.figure(figsize=(18, 10))
        fig2.patch.set_facecolor(C['bg'])
        gs = GridSpec(2, 3, figure=fig2,
                      hspace=0.42, wspace=0.32)

        # ─── Panel A: P&L Waterfall ───────────────────────────────────────────
        ax_wf = fig2.add_subplot(gs[0, :])
        ax_wf.set_facecolor(C['card'])

        wf_labels = [
            'Revenue', 'COGS', 'Phys Margin',
            'Freight', 'Terminal', 'BOG',
            'Brokerage+Other', 'EBITDA',
            'Tax', 'Hedge', 'NET P&L'
        ]
        wf_values = [
            r['revenue'], -r['cogs'], r['phys_margin'],
            -r['total_freight'], -(r['regas_cost'] + r['storage_cost']),
            -r['bog_cost'],
            -(r['broker_cost'] + r['other_cost']),
            r['ebitda'],
            -r['tax_amount'], r['hedge_gain'], r['net_profit']
        ]

        # Running totals for waterfall
        totals     = [True, False, True, False, False,
                      False, False, True, False, False, True]
        running    = 0
        bottoms    = []
        bar_colors = []

        for i, (v_val, is_total) in enumerate(zip(wf_values, totals)):
            if is_total:
                bottoms.append(0)
                bar_colors.append(
                    C['blue'] if i < len(wf_values) - 1
                    else (C['gain'] if v_val >= 0 else C['loss'])
                )
                running = v_val
            else:
                if v_val >= 0:
                    bottoms.append(running)
                    bar_colors.append(C['gain'])
                else:
                    bottoms.append(running + v_val)
                    bar_colors.append(C['loss'])
                running += v_val

        x = np.arange(len(wf_labels))
        bars = ax_wf.bar(x, [abs(v_val) for v_val in wf_values],
                         bottom=bottoms, color=bar_colors,
                         width=0.65, alpha=0.88,
                         edgecolor='#21262d', linewidth=0.5)

        # Value labels on bars
        scale = 1e6
        for bar, v_val in zip(bars, wf_values):
            ypos = bar.get_y() + bar.get_height() + \
                   abs(max(wf_values)) * 0.01
            ax_wf.text(bar.get_x() + bar.get_width() / 2,
                       ypos,
                       f'${v_val/1e6:+.2f}M',
                       ha='center', va='bottom',
                       fontsize=7.5, color=C['text'],
                       fontweight='bold')

        ax_wf.set_xticks(x)
        ax_wf.set_xticklabels(wf_labels,
                               fontsize=8.5, rotation=15)
        ax_wf.axhline(0, color=C['border'],
                      linewidth=0.8, linestyle='--')
        ax_wf.yaxis.set_major_formatter(
            mtick.FuncFormatter(lambda v, _: f'${v/1e6:.1f}M')
        )
        ax_wf.set_title('P&L Waterfall',
                         fontsize=11, pad=10)
        ax_wf.grid(True, axis='y', alpha=0.3)

        # ─── Panel B: Price Sensitivity ───────────────────────────────────────
        ax_sens = fig2.add_subplot(gs[1, 0])
        ax_sens.set_facecolor(C['card'])

        pnl_mm = [p / 1e6 for p in r['pnl_curve']]
        ax_sens.plot(r['price_range'], pnl_mm,
                     color=C['blue'], linewidth=2)
        ax_sens.fill_between(r['price_range'], pnl_mm, 0,
                              where=[p >= 0 for p in pnl_mm],
                              alpha=0.25, color=C['gain'])
        ax_sens.fill_between(r['price_range'], pnl_mm, 0,
                              where=[p < 0 for p in pnl_mm],
                              alpha=0.25, color=C['loss'])
        ax_sens.axhline(0, color=C['border'],
                         linewidth=1, linestyle='--')
        ax_sens.axvline(r['sell_px'],
                        color=C['blue'], linewidth=1.5,
                        linestyle=':', alpha=0.8,
                        label=f'Current: ${r["sell_px"]:.2f}')
        ax_sens.axvline(r['breakeven_sell'],
                        color=C['warn'], linewidth=1.5,
                        linestyle=':', alpha=0.8,
                        label=f'B/E: ${r["breakeven_sell"]:.2f}')
        ax_sens.set_xlabel('Sell Price ($/MMBtu)', fontsize=8)
        ax_sens.yaxis.set_major_formatter(
            mtick.FuncFormatter(lambda v, _: f'${v:.1f}M')
        )
        ax_sens.set_title('P&L vs Sell Price', fontsize=10)
        ax_sens.legend(fontsize=7.5)
        ax_sens.grid(True, alpha=0.3)

        # ─── Panel C: Cost Breakdown Pie ──────────────────────────────────────
        ax_pie = fig2.add_subplot(gs[1, 1])
        ax_pie.set_facecolor(C['card'])

        cost_items = {
            'COGS':      r['cogs'],
            'Freight':   r['total_freight'],
            'Regas':     r['regas_cost'],
            'BOG':       r['bog_cost'],
            'Storage':   r['storage_cost'],
            'Brokerage': r['broker_cost'],
            'Other':     r['other_cost'],
        }
        cost_items = {k: v for k, v in cost_items.items()
                      if v > 0}
        pie_colors = [
            C['loss'], C['warn'], C['blue'],
            C['purple'], C['teal'], C['muted'], '#444'
        ]

        wedges, texts, autotexts = ax_pie.pie(
            list(cost_items.values()),
            labels=list(cost_items.keys()),
            colors=pie_colors[:len(cost_items)],
            autopct='%1.0f%%',
            startangle=140,
            pctdistance=0.80,
            textprops={'color': C['text'], 'fontsize': 8}
        )
        for at in autotexts:
            at.set_color(C['bg'])
            at.set_fontsize(7)
            at.set_fontweight('bold')
        ax_pie.set_title('Cost Breakdown', fontsize=10)

        # ─── Panel D: Stress Test Bar ─────────────────────────────────────────
        ax_stress = fig2.add_subplot(gs[1, 2])
        ax_stress.set_facecolor(C['card'])

        scenarios = {
            'Base Case':        r['net_profit'],
            'Price −$2':        r['net_profit'] + (-2 * r['net']),
            'Price +$2':        r['net_profit'] + (2  * r['net']),
            'Freight +50%':     r['net_profit'] - r['total_freight'] * 0.5,
            'BOG ×2':           r['net_profit'] - r['bog_cost'],
            'User Stress':      r['s_net'],
        }
        s_labels = list(scenarios.keys())
        s_vals   = [v / 1e6 for v in scenarios.values()]
        s_colors = [C['gain'] if v >= 0 else C['loss']
                    for v in s_vals]

        bars_s = ax_stress.barh(s_labels, s_vals,
                                 color=s_colors, height=0.55,
                                 alpha=0.85,
                                 edgecolor='#21262d',
                                 linewidth=0.5)
        for bar, val in zip(bars_s, s_vals):
            xpos = val + (max(s_vals) - min(s_vals)) * 0.02
            ax_stress.text(xpos,
                           bar.get_y() + bar.get_height() / 2,
                           f'${val:+.2f}M',
                           va='center', fontsize=7.5,
                           color=C['text'])
        ax_stress.axvline(0, color=C['border'],
                           linewidth=0.8, linestyle='--')
        ax_stress.xaxis.set_major_formatter(
            mtick.FuncFormatter(lambda v, _: f'${v:.1f}M')
        )
        ax_stress.set_title('Stress Test Scenarios',
                             fontsize=10)
        ax_stress.grid(True, axis='x', alpha=0.3)

        fig2.suptitle('LNG Deal Analysis — Financial Detail',
                       fontsize=13, fontweight='bold',
                       color=C['text'], y=1.01)
        plt.tight_layout()
        plt.show()

        # ══════════════════════════════════════════════════════════════════════
        # FIG 3 — RISK & PRICING DETAIL
        # ══════════════════════════════════════════════════════════════════════
        fig3, (ax_var, ax_sc, ax_mtm) = plt.subplots(
            1, 3, figsize=(18, 5)
        )
        fig3.patch.set_facecolor(C['bg'])

        # ─── VaR Distribution ─────────────────────────────────────────────────
        ax_var.set_facecolor(C['card'])
        np.random.seed(42)
        sim_returns = np.random.normal(
            0,
            w_price_vol.value / 100 * r['sell_px'],
            5000
        )
        sim_pnl = sim_returns * r['unhedged_vol'] / 1e6
        var_line = -r['var_1d'] / 1e6

        n, bins, patches = ax_var.hist(
            sim_pnl, bins=60, alpha=0.75,
            color=C['blue'], edgecolor='none'
        )
        for p, b in zip(patches, bins):
            if b <= var_line:
                p.set_facecolor(C['loss'])

        ax_var.axvline(var_line,
                        color=C['loss'],
                        linewidth=2, linestyle='--',
                        label=f'{w_var_conf.value}% VaR: ${-var_line:.2f}M')
        ax_var.axvline(0, color=C['border'],
                        linewidth=0.8, linestyle=':')
        ax_var.set_xlabel('Daily P&L ($M)', fontsize=9)
        ax_var.set_ylabel('Frequency', fontsize=9)
        ax_var.set_title('VaR — P&L Distribution\n'
                          f'(Unhedged exposure: '
                          f'{r["unhedged_vol"]/1e6:.1f}M MMBtu)',
                          fontsize=10)
        ax_var.legend(fontsize=8)
        ax_var.grid(True, alpha=0.3)

        # ─── S-Curve Chart ─────────────────────────────────────────────────────
        ax_sc.set_facecolor(C['card'])

        if w_buy_type.value == 'JCC Slope':
            ax_sc.plot(r['jcc_range'], r['formula_price'],
                       color=C['muted'], linewidth=1.5,
                       linestyle='--', label='Linear (no S-curve)',
                       alpha=0.6)
            ax_sc.plot(r['jcc_range'], r['scurve_p'],
                       color=C['blue'], linewidth=2.5,
                       label='S-Curve Price')
            ax_sc.axvline(w_jcc.value,
                           color=C['warn'], linewidth=1.5,
                           linestyle=':',
                           label=f'Current JCC: ${w_jcc.value:.1f}')
            ax_sc.axvline(w_scurve_cap.value,
                           color=C['loss'], linewidth=1,
                           linestyle=':', alpha=0.7,
                           label=f'Cap: ${w_scurve_cap.value:.0f}')
            ax_sc.axvline(w_scurve_floor.value,
                           color=C['gain'], linewidth=1,
                           linestyle=':', alpha=0.7,
                           label=f'Floor: ${w_scurve_floor.value:.0f}')
            ax_sc.set_xlabel('JCC ($/bbl)', fontsize=9)
            ax_sc.set_ylabel('LNG Price ($/MMBtu)', fontsize=9)
            ax_sc.set_title('JCC Pricing S-Curve', fontsize=10)
            ax_sc.legend(fontsize=7.5)
        else:
            # Show price vs margin for non-JCC
            test_range = np.linspace(
                r['sell_px'] * 0.5,
                r['sell_px'] * 1.5, 100
            )
            margins = [(p - r['buy_px']) for p in test_range]
            ax_sc.plot(test_range, margins,
                       color=C['blue'], linewidth=2)
            ax_sc.axhline(0, color=C['border'],
                           linewidth=0.8, linestyle='--')
            ax_sc.axvline(r['sell_px'],
                           color=C['warn'], linewidth=1.5,
                           linestyle=':',
                           label=f'Current: ${r["sell_px"]:.2f}')
            ax_sc.fill_between(test_range, margins, 0,
                                where=[m >= 0 for m in margins],
                                alpha=0.2, color=C['gain'])
            ax_sc.fill_between(test_range, margins, 0,
                                where=[m < 0 for m in margins],
                                alpha=0.2, color=C['loss'])
            ax_sc.set_xlabel('Sell Price ($/MMBtu)', fontsize=9)
            ax_sc.set_ylabel('Gross Margin ($/MMBtu)', fontsize=9)
            ax_sc.set_title('Gross Margin vs Sell Price', fontsize=10)
            ax_sc.legend(fontsize=7.5)
        ax_sc.grid(True, alpha=0.3)

        # ─── BOG & Volume Waterfall ───────────────────────────────────────────
        ax_mtm.set_facecolor(C['card'])

        vol_labels = ['Loaded\n(Gross)',
                      'BOG\nLoss',
                      'Heel\nRetained',
                      'Net\nDelivered']
        vol_vals   = [r['v']['gross'],
                      -r['v']['bog'],
                      -r['v']['heel'],
                      r['v']['net']]
        vol_bots   = [0,
                      r['v']['gross'] - r['v']['bog'],
                      r['v']['gross'],
                      0]
        vol_colors = [C['blue'], C['loss'],
                      C['warn'],
                      C['gain'] if r['v']['net'] > 0 else C['loss']]
        # Adjust bottoms for waterfall
        vol_bots = [
            0,
            r['v']['gross'] + vol_vals[1],  # gross - bog
            r['v']['gross'] + vol_vals[1],  # same level, heel reduces further
            0
        ]

        b_bars = ax_mtm.bar(
            vol_labels,
            [abs(v) for v in vol_vals],
            bottom=vol_bots,
            color=vol_colors,
            width=0.55, alpha=0.88,
            edgecolor='#21262d', linewidth=0.5
        )
        for bar, val in zip(b_bars, vol_vals):
            yp = bar.get_y() + bar.get_height() + \
                 r['v']['gross'] * 0.01
            ax_mtm.text(
                bar.get_x() + bar.get_width() / 2,
                yp,
                f'{abs(val)/1e6:.2f}M',
                ha='center', va='bottom',
                fontsize=8, color=C['text'],
                fontweight='bold'
            )
        ax_mtm.set_ylabel('MMBtu', fontsize=9)
        ax_mtm.set_title(
            f'Volume Waterfall (BOG: {r["v"]["bog_pct"]:.2f}% lost)',
            fontsize=10
        )
        ax_mtm.yaxis.set_major_formatter(
            mtick.FuncFormatter(lambda v, _:
                                f'{v/1e6:.1f}M')
        )
        ax_mtm.grid(True, axis='y', alpha=0.3)

        fig3.suptitle('Risk Analytics & Pricing Detail',
                       fontsize=13, fontweight='bold',
                       color=C['text'])
        plt.tight_layout()
        plt.show()

        # ══════════════════════════════════════════════════════════════════════
        # DETAILED TEXT REPORT
        # ══════════════════════════════════════════════════════════════════════
        print(f"""
{'═'*70}
 LNG DEAL FEASIBILITY REPORT
 Generated: {date.today().strftime('%d %b %Y')}
{'═'*70}

CARGO & VOLUME
  Vessel Capacity:       {w_volume.value:>12,.0f}  m³
  Weight:                {r['v']['weight']:>12,.0f}  tonnes
  Gross Energy:          {r['v']['gross']:>12,.0f}  MMBtu
  BOG Loss ({w_voyage_days.value} days):    {r['v']['bog']:>12,.0f}  MMBtu ({r['v']['bog_pct']:.2f}%)
  Heel Retained:         {r['v']['heel']:>12,.0f}  MMBtu
  NET DELIVERED:         {r['net']:>12,.0f}  MMBtu

PRICING
  Buy Price ({w_buy_type.value}): ${r['buy_px']:>8.4f} /MMBtu
  Sell Price ({w_sell_type.value}): ${r['sell_px']:>8.4f} /MMBtu
  Raw Spread:            ${r['sell_px'] - r['buy_px']:>+8.4f} /MMBtu
  Break-Even Sell Px:    ${r['breakeven_sell']:>8.4f} /MMBtu
  Headroom to Break-Even:${r['be_margin']:>+8.4f} /MMBtu

P&L WATERFALL
  Revenue:               ${r['revenue']:>14,.0f}
  Cost of Cargo (COGS):  ${-r['cogs']:>14,.0f}
  ─────────────────────────────────────────────
  Physical Margin:       ${r['phys_margin']:>+14,.0f}

  Freight (variable):    ${-r['freight_var']:>14,.0f}
  Freight (fixed):       ${-r['freight_fix']:>14,.0f}
  Regasification:        ${-r['regas_cost']:>14,.0f}
  Storage:               ${-r['storage_cost']:>14,.0f}
  BOG Expense:           ${-r['bog_cost']:>14,.0f}
  Brokerage:             ${-r['broker_cost']:>14,.0f}
  Other Costs:           ${-r['other_cost']:>14,.0f}
  ─────────────────────────────────────────────
  EBITDA:                ${r['ebitda']:>+14,.0f}
  Tax ({w_tax_rate.value:.0f}%):             ${-r['tax_amount']:>14,.0f}
  Hedge P&L:             ${r['hedge_gain']:>+14,.0f}
  ─────────────────────────────────────────────
  NET PROFIT:            ${r['net_profit']:>+14,.0f}
  Per MMBtu:             ${r['margin_mmbtu']:>+14.4f}
  NPV:                   ${r['npv']:>+14,.0f}
  Return on Capital:     {r['roc']:>+13.2f}%

RISK METRICS
  Hedge Ratio:           {w_hedge_ratio.value:>12.0f}%
  Unhedged Volume:       {r['unhedged_vol']:>12,.0f}  MMBtu
  VaR ({w_var_conf.value}%, 1-day):      ${r['var_1d']:>14,.0f}
  VaR ({w_var_conf.value}%, {w_var_horizon.value}-day):      ${r['var_nd']:>14,.0f}
  Credit Utilization:    {r['credit_util']:>12.1f}%

STRESS TEST (User Defined)
  JKM Shock:             ${w_stress_jkm.value:>+12.2f} /MMBtu
  Freight Shock:         ${w_stress_freight.value:>+12.2f} /MMBtu
  FX Move:               {w_stress_fx.value:>+12.1f}%
  Stressed Net P&L:      ${r['s_net']:>+14,.0f}
  P&L Delta vs Base:     ${r['stress_delta']:>+14,.0f}

{'═'*70}
 VERDICT:  {verdict}
{'═'*70}
""")

# ── Run button ────────────────────────────────────────────────────────────────
run_btn = widgets.Button(
    description='▶  Run Full Analysis',
    button_style='success',
    layout=widgets.Layout(width='260px', height='44px')
)
run_btn.on_click(generate_report)

update_btn = widgets.Button(
    description='🔄  Refresh (same inputs)',
    button_style='info',
    layout=widgets.Layout(width='220px', height='44px')
)
update_btn.on_click(generate_report)

display(widgets.HBox([run_btn, update_btn],
                     layout=widgets.Layout(gap='16px',
                                           padding='8px')))
display(report_out)

Output()

---
## 🔄 Section 7 — Multi-Scenario Comparison
*Define up to four named scenarios and compare P&L side-by-side.*

In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 8 — SCENARIO COMPARISON
# ─────────────────────────────────────────────────────────────────────────────

s_layout = widgets.Layout(width='200px')
s_style  = {'description_width': '110px'}

def make_scenario_inputs(label, jkm_d, sell_d, freight_d, voyage_d):
    return {
        'label': widgets.Text(
            value=label,
            description='Name:',
            style=s_style, layout=s_layout
        ),
        'jkm_delta': widgets.FloatText(
            value=jkm_d,
            description='JKM Δ:',
            style=s_style, layout=s_layout
        ),
        'sell_delta': widgets.FloatText(
            value=sell_d,
            description='Sell Δ:',
            style=s_style, layout=s_layout
        ),
        'freight_delta': widgets.FloatText(
            value=freight_d,
            description='Freight Δ:',
            style=s_style, layout=s_layout
        ),
        'voyage_days_delta': widgets.IntText(
            value=voyage_d,
            description='Voyage Δ days:',
            style=s_style, layout=s_layout
        ),
    }

scenarios = [
    make_scenario_inputs('Base Case',     0.0,  0.0,  0.0, 0),
    make_scenario_inputs('Bull (JKM+$2)', 2.0,  2.0,  0.0, 0),
    make_scenario_inputs('Bear (JKM−$3)',-3.0, -3.0,  0.0, 0),
    make_scenario_inputs('Ship+Slow',     0.0,  0.0,  0.8, 5),
]

scenario_boxes = []
for sc in scenarios:
    box = widgets.VBox(
        [sc['label'], sc['jkm_delta'], sc['sell_delta'],
         sc['freight_delta'], sc['voyage_days_delta']],
        layout=widgets.Layout(
            padding='10px',
            border='1px solid #30363d',
            border_radius='6px',
            width='220px'
        )
    )
    scenario_boxes.append(box)

compare_out = widgets.Output()

def run_scenarios(_=None):
    base = compute_all()
    results_list = []

    for sc in scenarios:
        # Temporarily shift values
        orig_jkm_buy   = w_jkm_buy.value
        orig_jkm_sell  = w_jkm_sell.value
        orig_ttf       = w_ttf_sell.value
        orig_freight   = w_freight.value
        orig_voyage    = w_voyage_days.value

        w_jkm_buy.value   = orig_jkm_buy  + sc['jkm_delta'].value
        w_jkm_sell.value  = orig_jkm_sell + sc['sell_delta'].value
        w_ttf_sell.value  = orig_ttf + sc['sell_delta'].value
        w_freight.value   = max(
            0.1, orig_freight + sc['freight_delta'].value
        )
        w_voyage_days.value = max(
            1, orig_voyage + sc['voyage_days_delta'].value
        )

        r = compute_all()
        results_list.append({
            'name':       sc['label'].value,
            'net_profit': r['net_profit'],
            'margin':     r['margin_mmbtu'],
            'revenue':    r['revenue'],
            'total_costs':r['total_costs'] + r['cogs'],
            'var':        r['var_1d'],
            'roc':        r['roc'],
            'breakeven':  r['breakeven_sell'],
        })

        # Restore
        w_jkm_buy.value     = orig_jkm_buy
        w_jkm_sell.value    = orig_jkm_sell
        w_ttf_sell.value    = orig_ttf
        w_freight.value     = orig_freight
        w_voyage_days.value = orig_voyage

    with compare_out:
        clear_output(wait=True)

        fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
        fig.patch.set_facecolor(C['bg'])

        names  = [r['name'] for r in results_list]
        colors = [
            C['gain'] if r['net_profit'] >= 0 else C['loss']
            for r in results_list
        ]

        metrics = [
            ('Net P&L ($M)',
             [r['net_profit'] / 1e6 for r in results_list],
             '${:.2f}M'),
            ('Margin ($/MMBtu)',
             [r['margin'] for r in results_list],
             '${:+.3f}'),
            ('Return on Capital (%)',
             [r['roc'] for r in results_list],
             '{:+.1f}%'),
            ('1-Day VaR ($M)',
             [r['var'] / 1e6 for r in results_list],
             '${:.2f}M'),
        ]

        for ax, (title, vals, fmt) in zip(axes, metrics):
            ax.set_facecolor(C['card'])
            bar_c = [
                C['gain'] if v >= 0 else C['loss']
                for v in vals
            ]
            if title == '1-Day VaR ($M)':
                bar_c = [C['warn']] * len(vals)

            b = ax.bar(names, vals, color=bar_c,
                       alpha=0.85, edgecolor='#21262d',
                       linewidth=0.5)
            for bar, val in zip(b, vals):
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_height() +
                    max(abs(v) for v in vals) * 0.02,
                    fmt.format(val),
                    ha='center', va='bottom',
                    fontsize=8.5, color=C['text'],
                    fontweight='bold'
                )
            ax.axhline(0, color=C['border'],
                       linewidth=0.8, linestyle='--')
            ax.set_title(title, fontsize=10)
            ax.set_xticklabels(names, fontsize=8,
                                rotation=12)
            ax.grid(True, axis='y', alpha=0.3)

        fig.suptitle('Scenario Comparison',
                      fontsize=13, fontweight='bold',
                      color=C['text'])
        plt.tight_layout()
        plt.show()

        # Table
        df_sc = pd.DataFrame(results_list).set_index('name')
        df_sc['net_profit'] = df_sc['net_profit'].map(
            '${:+,.0f}'.format
        )
        df_sc['margin']     = df_sc['margin'].map(
            '${:+.4f}/MMBtu'.format
        )
        df_sc['roc']        = df_sc['roc'].map(
            '{:+.2f}%'.format
        )
        df_sc['var']        = df_sc['var'].map(
            '${:,.0f}'.format
        )
        df_sc['breakeven']  = df_sc['breakeven'].map(
            '${:.4f}'.format
        )
        df_sc.columns = [
            'Net P&L', 'Margin/MMBtu',
            'Revenue', 'Total Costs',
            'VaR (1d)', 'ROC', 'Break-Even'
        ]
        display(df_sc[[
            'Net P&L', 'Margin/MMBtu',
            'VaR (1d)', 'ROC', 'Break-Even'
        ]])

compare_btn = widgets.Button(
    description='▶  Compare Scenarios',
    button_style='warning',
    layout=widgets.Layout(width='220px', height='42px')
)
compare_btn.on_click(run_scenarios)

display(widgets.HTML('<p style="color:#8b949e;font-size:11px">'
                     'Define 4 scenarios — delta values shift from '
                     'current Section 2–3 inputs.</p>'))
display(widgets.HBox(scenario_boxes,
                     layout=widgets.Layout(gap='10px')))
display(compare_btn)
display(compare_out)

HTML(value='<p style="color:#8b949e;font-size:11px">Define 4 scenarios — delta values shift from current Secti…

Button(button_style='warning', description='▶  Compare Scenarios', layout=Layout(height='42px', width='220px')…

Output()

---
## 📋 Section 8 — Export Deal Sheet
*Export all inputs and results as a structured JSON deal sheet.*

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 9 — EXPORT
# ─────────────────────────────────────────────────────────────────────────────

import base64
export_out = widgets.Output()

def export_deal(_=None):
    r = compute_all()

    deal_sheet = {
        'meta': {
            'generated_date': str(date.today()),
            'tool': 'LNG Deal Feasibility Analyser',
            'verdict': ('FEASIBLE' if r['net_profit'] > 0
                        else 'MARGINAL'
                        if r['net_profit'] > -r['peak_cash'] * 0.02
                        else 'NOT_FEASIBLE')
        },
        'cargo': {
            'vessel_capacity_m3':  w_volume.value,
            'lng_density_t_m3':    w_density.value,
            'cv_mmbtu_per_tonne':  w_cv.value,
            'bog_rate_pct_per_day': w_bog_rate.value,
            'voyage_days':         w_voyage_days.value,
            'heel_m3':             w_heel.value,
        },
        'volumes': {
            'weight_tonnes':        round(r['v']['weight'], 0),
            'gross_mmbtu':          round(r['v']['gross'], 0),
            'bog_loss_mmbtu':       round(r['v']['bog'], 0),
            'heel_mmbtu':           round(r['v']['heel'], 0),
            'net_delivered_mmbtu':  round(r['net'], 0),
            'bog_loss_pct':         round(r['v']['bog_pct'], 3),
        },
        'pricing': {
            'buy_type':            w_buy_type.value,
            'sell_type':           w_sell_type.value,
            'buy_price_mmbtu':     round(r['buy_px'], 4),
            'sell_price_mmbtu':    round(r['sell_px'], 4),
            'raw_spread_mmbtu':    round(r['sell_px'] - r['buy_px'], 4),
            'breakeven_sell':      round(r['breakeven_sell'], 4),
            'headroom_mmbtu':      round(r['be_margin'], 4),
            'fx_rate':             w_fx.value,
        },
        'pnl': {
            'revenue':             round(r['revenue'], 0),
            'cogs':                round(r['cogs'], 0),
            'physical_margin':     round(r['phys_margin'], 0),
            'total_freight':       round(r['total_freight'], 0),
            'regas_storage':       round(r['regas_cost']+r['storage_cost'], 0),
            'bog_expense':         round(r['bog_cost'], 0),
            'other_costs':         round(r['broker_cost']+r['other_cost'], 0),
            'ebitda':              round(r['ebitda'], 0),
            'tax':                 round(r['tax_amount'], 0),
            'hedge_pnl':           round(r['hedge_gain'], 0),
            'net_profit':          round(r['net_profit'], 0),
            'net_profit_per_mmbtu': round(r['margin_mmbtu'], 4),
            'npv':                 round(r['npv'], 0),
            'roc_pct':             round(r['roc'], 3),
        },
        'risk': {
            'hedge_ratio_pct':     w_hedge_ratio.value,
            'hedge_price_mmbtu':   w_hedge_price.value,
            'var_1d_usd':          round(r['var_1d'], 0),
            'var_nd_usd':          round(r['var_nd'], 0),
            'var_confidence_pct':  w_var_conf.value,
            'var_horizon_days':    w_var_horizon.value,
            'credit_utilization_pct': round(r['credit_util'], 2),
        },
        'stress': {
            'jkm_shock_mmbtu':     w_stress_jkm.value,
            'freight_shock_mmbtu': w_stress_freight.value,
            'fx_move_pct':         w_stress_fx.value,
            'stressed_net_profit': round(r['s_net'], 0),
            'pnl_delta_vs_base':   round(r['stress_delta'], 0),
        }
    }

    fname = f"lng_deal_{date.today().strftime('%Y%m%d')}.json"
    json_str = json.dumps(deal_sheet, indent=2)

    # Write file to disk
    with open(fname, 'w') as f:
        f.write(json_str)

    with export_out:
        clear_output(wait=True)
        print(f'\n✅  Deal sheet exported to: {fname}')
        print(f'    Verdict: {deal_sheet["meta"]["verdict"]}')
        print(f'    Net P&L: ${deal_sheet["pnl"]["net_profit"]:+,.0f}')
        print(f'    ROC:     {deal_sheet["pnl"]["roc_pct"]:+.2f}%')
        print('\n--- JSON Preview (first 30 lines) ---')
        lines = json_str.split('\n')
        print('\n'.join(lines[:30]) + '\n  ...')

    # Trigger browser download
    if IN_COLAB:
        colab_files.download(fname)
    else:
        # Local Jupyter — render a clickable download link
        b64 = base64.b64encode(json_str.encode()).decode()
        dl_link = (
            f'<a download="{fname}" '
            f'href="data:application/json;base64,{b64}" '
            f'style="font-size:14px;color:#58a6ff;">'
            f'⬇ Click here to download {fname}</a>'
        )
        display(HTML(dl_link))

export_btn = widgets.Button(
    description='💾  Export Deal Sheet (JSON)',
    button_style='',
    layout=widgets.Layout(width='260px', height='42px')
)
export_btn.on_click(export_deal)

display(widgets.HTML(
    '<p style="color:#8b949e;font-size:11px">'
    'Exports all current inputs and computed results '
    'to a JSON file. In Colab a download dialog appears; '
    'in local Jupyter a download link is shown.</p>'
))
display(export_btn)
display(export_out)

HTML(value='<p style="color:#8b949e;font-size:11px">Exports all current inputs and computed results to a JSON …

Button(description='💾  Export Deal Sheet (JSON)', layout=Layout(height='42px', width='260px'), style=ButtonSty…

Output()

---
### 📌 Quick Reference — Key Inputs

| Parameter | Typical Range | Notes |
|---|---|---|
| Vessel Capacity | 65,000 – 266,000 m³ | Standard = 138,000–155,000 m³ |
| LNG Density | 0.42 – 0.50 t/m³ | Depends on gas composition |
| Calorific Value | 19.0 – 23.5 MMBtu/t | Higher = richer gas |
| BOG Rate | 0.08% – 0.15%/day | Modern vessels at lower end |
| JCC Slope | 0.13 – 0.16 | Long-term Asia SPA norm |
| JKM Spot | $8 – $20/MMBtu | Market dependent |
| TTF Equivalent | $6 – $25/MMBtu | European benchmark |
| Voyage Charter | $1.50 – $5.00/MMBtu | Spot rate volatile |
| Regasification | $0.20 – $0.80/MMBtu | By terminal type |
| Hedge Ratio | 50% – 100% | Trader's market view |
| VaR Confidence | 95% – 99% | Risk appetite |

---
*LNG Deal Feasibility Analyser — Built for PacificLNG Trading Co.*